In [1]:
import pandas as pd
from pathlib import Path

RAW  = Path("../data/raw")
PROC = Path("../data/processed")

# 1. PAM50 labels
lab = pd.read_csv(RAW / "TCGA_BRCA_subtypes.csv")
lab.columns = ["aliquot", "subtype"]
lab["subtype"]    = lab["subtype"].str.replace("BRCA_", "", regex=False)
lab["sample_id"]  = lab["aliquot"].str[:16]
lab["sample_key"] = lab["aliquot"].str[:15]
lab["patient_id"] = lab["aliquot"].str[:12]

# 2. One sample per patient (prefer vial A)
lab = lab.drop_duplicates("sample_id")
lab["priority"] = (lab["sample_id"].str[-1] != "A").astype(int)
lab = (lab.sort_values(["patient_id", "priority"])
          .drop_duplicates("patient_id").drop(columns="priority"))

count_cols = pd.read_csv(PROC / "TCGA_BRCA_raw_counts.csv", nrows=0).columns
lab = lab[lab["sample_id"].isin(count_cols)].reset_index(drop=True)
print("Patients:", len(lab)); print(lab["subtype"].value_counts())

# 3. Clinical
clin = pd.read_csv(RAW / "BRCA_clinicalMatrix.tsv", sep="\t", low_memory=False)
clin = clin.rename(columns={
    "sampleID": "sample_key",
    "age_at_initial_pathologic_diagnosis": "age",
    "pathologic_stage": "stage",
    "breast_carcinoma_estrogen_receptor_status": "ER",
    "breast_carcinoma_progesterone_receptor_status": "PR",
    "lab_proc_her2_neu_immunohistochemistry_receptor_status": "HER2_IHC",
})[["sample_key", "age", "gender", "stage", "ER", "PR", "HER2_IHC"]]

# 4. Survival
pc = pd.read_csv(RAW / "brca_tcga_pan_can_atlas_2018_clinical_data.tsv", sep="\t")
surv = pd.DataFrame({
    "sample_key": pc["Sample ID"],
    "OS_months":  pc["Overall Survival (Months)"],
    "OS_event":   pc["Overall Survival Status"].str[0].astype(float),
})

meta = (lab.merge(clin, on="sample_key", how="left", validate="one_to_one")
           .merge(surv, on="sample_key", how="left", validate="one_to_one"))
print("\nMissing values:\n", meta.isna().sum())

# 5. Count matrix
counts = pd.read_csv(PROC / "TCGA_BRCA_raw_counts.csv", index_col="gene_id",
                     usecols=["gene_id"] + meta["sample_id"].tolist())
counts = counts[meta["sample_id"]]
print("\nCounts (genes x samples):", counts.shape)

# 6. Save
meta.to_csv(PROC / "BRCA_pam50_meta.csv", index=False)
counts.to_csv(PROC / "BRCA_pam50_counts.csv")
print("Saved.")

Patients: 1083
subtype
LumA      562
LumB      209
Basal     190
Her2       82
Normal     40
Name: count, dtype: int64

Missing values:
 aliquot         0
subtype         0
sample_id       0
sample_key      0
patient_id      0
age             1
gender          1
stage           6
ER             49
PR             50
HER2_IHC      177
OS_months      14
OS_event       14
dtype: int64

Counts (genes x samples): (60660, 1083)
Saved.
